# Week 6 Predicting frozen risk profile membership

## Objective 2

**Research question:** how well can demographic, socioeconomic, discrimination, support and measured country-context variables predict membership in the frozen B4 and E8 profiles, including in countries outside supervised model training?

The targets are the existing Objective 1 assignments. We do not refit the clustering, rename its labels or assign a profile to excluded respondents. B4 is the joint worry/harm description; E8 is the complementary thematic worry description. We predict these two targets separately.

This first draft compares a training-majority baseline, multinomial logistic regression and a random forest with fixed settings. It uses five country-grouped development folds, then selects and seals a supervised specification before its held-out test. We retain fitted preprocessing, predictions, aggregate tables, figures and failures in a fingerprint-specific result folder.

**Reading order:** establish the target and predictor definitions; audit missingness and exclusions; compare development performance; freeze the supervised choice; inspect development predictor associations; evaluate held-out performance and geographic variation; draw qualified conclusions.

**Interpretation:** we predict a fixed descriptive classification, not an underlying diagnosis or a causal social type. Weak background prediction would not by itself invalidate Objective 1. Strong prediction would not establish causation.

The full cohort and stored cluster identifiers are authoritative. Display profile 1 on the webpage corresponds to saved C0; labels such as B_C0 and E_C0 are preserved here.

## 1 Execution controls and reproducible implementation

**Objective:** make expensive or test-data operations explicit and prevent accidental re-fitting on notebook reopening.

The code below is embedded in this notebook. It depends on the same scientific environment and local prepared data used in Weeks 3–5, not on an absolute personal path. Execution switches are saved as false; the completed first-draft outputs can be regenerated from verified local fit caches. Set a switch to true only when intentionally computing a missing stage. The supervised held-out switch operates only after the saved development seal exists.

The context includes hashes of the two prepared datasets, dictionary, country split, frozen assignments, Objective 1 seal and implementation. Changed inputs or code use a new result context rather than silently reusing old fits. Cache hashes are checked before loading. Load executable objects only from this trusted local course folder.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown, Image

def find_project_root():
    candidates=[Path.cwd(),*Path.cwd().parents]
    return next(p for p in candidates if (p/'course_artifacts/week03_country_split.csv').exists())

ROOT=find_project_root()
RUN_DEVELOPMENT=False
RUN_INTERPRETATION=False
RUN_HELDOUT=False
IMPLEMENTATION_SHA256="9e4be5092a4d7c2c356ca34ab2cbf245d53d56c4c86aa319393062fd730d4fda"
"""Week 6 Objective 2: predict frozen profiles from external background variables.

This module never fits clusters, changes their labels or imputes missing targets.
All respondent-level outputs and executable fit objects are local protected results.
"""
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, log_loss
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

FREEZE='123d248adda6115f'
FEATURE_BLOCKS={
    'Demographics and household':['Age','Gender','Education','Urbanicity2','EMP_2010',
        'INCOME_5','HouseholdSize','ChildrenInHousehold','WP1223'],
    'Financial buffer':['WP22228'],
    'Government and neighbour support':['WP22231_ALL','WP22232'],
    'Discrimination':['WP22259','WP22260','WP22261','WP22262','WP22263'],
    'Country income':['CountryIncomeLevel'],
}
PREDICTORS=[c for columns in FEATURE_BLOCKS.values() for c in columns]
NUMERIC=['Age']
CATEGORICAL=[c for c in PREDICTORS if c not in NUMERIC]
VALID_CODES={'Gender':[1,2],'Education':[1,2,3],'Urbanicity2':[1,2],
    'EMP_2010':[1,2,3,4,5,6],'INCOME_5':[1,2,3,4,5],
    'HouseholdSize':[1,2,3,4],'ChildrenInHousehold':[0,1,2,3,4,5],
    'WP1223':[1,2,3,4,5,8],'WP22228':[1,2],
    'WP22231_ALL':[1,2,3],'WP22232':[1,2,3],
    **{c:[1,2,97] for c in FEATURE_BLOCKS['Discrimination']},
    'CountryIncomeLevel':[1,2,3,4,9]}
MISSING_CODES={'Education':[9],'Urbanicity2':[9],'HouseholdSize':[9],
    'ChildrenInHousehold':[9],'WP1223':[6,7],'WP22228':[98,99],
    'WP22231_ALL':[99],'WP22232':[98,99],
    **{c:[98,99] for c in FEATURE_BLOCKS['Discrimination']}}
DESIGN={'version':1,'seed':20261014,'country_folds':5,'predictors':PREDICTORS,
    'models':{'majority':{'strategy':'prior'},
        'logistic':{'solver':'lbfgs','C':1.0,'max_iter':2000,'tol':1e-6},
        'random_forest':{'n_estimators':120,'max_depth':14,'min_samples_leaf':50,
            'max_features':'sqrt','bootstrap':True,'class_weight':None}},
    'numeric_preprocessing':'training equal-country-weighted median; weighted mean/SD; fixed missing indicator',
    'categorical_preprocessing':'explicit missing category; training one-hot encoder; unknown levels ignored',
    'fitting_weights':'equal country, relative WGT retained, mean weight rescaled to one',
    'evaluation_weights':'equal country, relative WGT retained, recomputed within evaluation population',
    'selection':'largest mean development-fold macro-F1; exact ties prefer logistic then forest then majority',
    'tuning':'none; fixed first-draft settings, not an optimized model search',
    'interpretation':'3 joint block permutations per fold on selected model; within-country for individual blocks, across countries for income',
    'heldout_uncertainty':'1000 uniform country bootstrap draws; fixed predictions and models',
    'profile_targets':{'B':4,'E':8},'profile_refits':False,
    'holdout_role':'supervised test outside selection; already used for Objective 1 validation',
    'cv_caution':'clustering targets were developed using all development countries; this is prediction of that fixed taxonomy, not a fully nested discovery evaluation'}

def timestamp():
    return datetime.now(timezone.utc).isoformat()

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def json_write(value,path):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
    temp.replace(path)

def country_weights(frame):
    if not frame.WGT.gt(0).all() or not np.isfinite(frame.WGT).all():
        raise ValueError('Survey weights must be positive and finite.')
    w=frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')/frame.COUNTRY_ISO3.nunique()
    assert np.isclose(w.sum(),1)
    assert np.allclose(w.groupby(frame.COUNTRY_ISO3).sum(),1/frame.COUNTRY_ISO3.nunique())
    return w.to_numpy(float)

def initialise(root,implementation_hash):
    root=Path(root).resolve()
    paths={'development':'course_artifacts/week03_development_data.parquet',
        'heldout':'course_artifacts/week03_heldout_country_data.parquet',
        'assignments':f'course_artifacts/week05_objective1_handoff_v1/{FREEZE}/all_respondents_eligibility_and_frozen_profiles.parquet',
        'dictionary':'23_wrp/WRP_2023_data_dictionary.csv',
        'split':'course_artifacts/week03_country_split.csv',
        'freeze_seal':f'course_artifacts/week05_frozen_objective1_v1/{FREEZE}/freeze_seal.json'}
    inputs={key:{'path':relative,'sha256':file_hash(root/relative)} for key,relative in paths.items()}
    reuse=json.loads((root/f'course_artifacts/week05_objective1_handoff_v1/{FREEZE}/objective1_completion_and_reuse.json').read_text())
    assert inputs['assignments']['sha256']==reuse['files']['all_respondents_eligibility_and_frozen_profiles.parquet']
    assert inputs['freeze_seal']['sha256']==reuse['frozen_model_seal_sha256']
    context={'design':DESIGN,'inputs':inputs,'implementation_sha256':implementation_hash,
        'python':platform.python_version(),'sklearn':sklearn.__version__,'numpy':np.__version__,
        'pandas':pd.__version__,'frozen_cluster_context':reuse['freeze_context']}
    fingerprint=hashlib.sha256(json.dumps(context,sort_keys=True).encode()).hexdigest()
    out=root/'course_artifacts/week06_objective2_profile_prediction_v1'/fingerprint[:16]
    out.mkdir(parents=True,exist_ok=True)
    context['fingerprint']=fingerprint
    if (out/'context.json').exists():
        assert json.loads((out/'context.json').read_text())==context
    else:
        json_write(context,out/'context.json')
    dictionary=pd.read_csv(root/paths['dictionary'])
    manifest=dictionary.loc[dictionary.variable.isin(PREDICTORS),['variable','label','value_labels','derivation']].copy()
    manifest['block']=manifest.variable.map({c:b for b,cs in FEATURE_BLOCKS.items() for c in cs})
    manifest['modelling_type']=np.where(manifest.variable.eq('Age'),'continuous with missing indicator','categorical')
    manifest['missing_codes']=manifest.variable.map(lambda c:json.dumps(MISSING_CODES.get(c,[])))
    manifest.to_csv(out/'predictor_manifest.csv',index=False)
    assert len(manifest)==len(PREDICTORS)==18
    return context,out

def load_cohort(root,context,split):
    root=Path(root)
    raw_key='development' if split=='train' else 'heldout'
    source=root/context['inputs'][raw_key]['path']
    assert file_hash(source)==context['inputs'][raw_key]['sha256']
    columns=['WPID_RANDOM','COUNTRY_ISO3','GlobalRegion','WGT','country_split']+PREDICTORS
    raw=pd.read_parquet(source,columns=columns)
    registry=pd.read_parquet(root/context['inputs']['assignments']['path'])
    registry=registry.loc[registry.country_split.eq(split)].copy()
    assert raw.WPID_RANDOM.is_unique and registry.WPID_RANDOM.is_unique
    assert set(raw.WPID_RANDOM)==set(registry.WPID_RANDOM)
    keys=raw[['WPID_RANDOM','COUNTRY_ISO3','WGT','country_split']].merge(
        registry[['WPID_RANDOM','COUNTRY_ISO3','WGT','country_split']],on='WPID_RANDOM',validate='one_to_one',suffixes=('_raw','_registry'))
    assert keys.COUNTRY_ISO3_raw.eq(keys.COUNTRY_ISO3_registry).all()
    assert keys.country_split_raw.eq(keys.country_split_registry).all()
    assert np.array_equal(keys.WGT_raw,keys.WGT_registry)
    frame=raw.merge(registry[['WPID_RANDOM','complete_case','exclusion_reason','B_cluster','E_cluster',
        'B_profile_id','E_profile_id']],on='WPID_RANDOM',validate='one_to_one')
    for model,k in DESIGN['profile_targets'].items():
        assert frame.loc[frame.complete_case,model+'_cluster'].isin(range(k)).all()
        assert frame.loc[~frame.complete_case,model+'_cluster'].isna().all()
        for label in range(k):
            assert frame.loc[frame[model+'_cluster'].eq(label),model+'_profile_id'].eq(f'{model}_C{label}').all()
    audit=frame.groupby('exclusion_reason',dropna=False).size().rename('respondents').reset_index()
    audit['split']=split
    frame=frame.loc[frame.complete_case].copy().reset_index(drop=True)
    n,countries=(109488,113) if split=='train' else (27972,29)
    assert len(frame)==n and frame.COUNTRY_ISO3.nunique()==countries
    return frame,audit

def prepare_predictors(frame):
    """Deterministic code validation only; no data-dependent fitting here."""
    x=frame[PREDICTORS].copy()
    if not x.Age.dropna().between(15,99).all():
        raise ValueError('Unexpected prepared Age code; do not treat age 100 as substantive.')
    for column in CATEGORICAL:
        allowed=VALID_CODES[column]+MISSING_CODES.get(column,[])
        unexpected=sorted(set(x[column].dropna())-set(allowed))
        if unexpected:
            raise ValueError(f'Unexpected codes in {column}: {unexpected}')
        x[column]=x[column].replace(MISSING_CODES.get(column,[]),np.nan)
        x[column]=x[column].map(lambda value:'missing' if pd.isna(value) else str(int(value)))
    # 97 is not applicable, not an absence of discrimination. 9 is a real
    # unclassified country-income category, not a secretly imputed income group.
    return x

def predictor_missingness(frame,x,scope):
    w=country_weights(frame)
    rows=[]
    for column in PREDICTORS:
        missing=x[column].isna().to_numpy() if column in NUMERIC else x[column].eq('missing').to_numpy()
        country=frame[['COUNTRY_ISO3']].assign(missing=missing)
        rows.append({'scope':scope,'feature':column,'missing_count':int(missing.sum()),
            'missing_fraction':float(missing.mean()),'equal_country_missing_fraction':float(w@missing),
            'countries_all_missing':int(country.groupby('COUNTRY_ISO3').missing.all().sum()),
            'not_applicable_count':int(x[column].eq('97').sum()) if column in CATEGORICAL else 0})
    return pd.DataFrame(rows)

def fit_preprocessor(x,weights):
    values=x.Age.to_numpy(float)
    observed=np.isfinite(values)
    order=np.argsort(values[observed],kind='stable')
    sorted_values=values[observed][order]
    cumulative=np.cumsum(weights[observed][order])
    median=float(sorted_values[np.searchsorted(cumulative,cumulative[-1]/2)])
    filled=np.where(observed,values,median)
    mean=float(weights@filled/weights.sum())
    scale=float(np.sqrt(weights@((filled-mean)**2)/weights.sum())) or 1.
    encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
    encoder.fit(x[CATEGORICAL])
    return {'age_median':median,'age_mean':mean,'age_scale':scale,'encoder':encoder,
        'predictors':PREDICTORS.copy(),'categorical':CATEGORICAL.copy()}

def transform_predictors(x,preprocessor):
    assert list(x.columns)==PREDICTORS
    age=x.Age.to_numpy(float)
    missing=~np.isfinite(age)
    filled=np.where(missing,preprocessor['age_median'],age)
    scaled=(filled-preprocessor['age_mean'])/preprocessor['age_scale']
    matrix=np.column_stack([scaled,missing.astype(float),preprocessor['encoder'].transform(x[CATEGORICAL])]).astype(np.float32)
    assert np.isfinite(matrix).all()
    return matrix

def build_model(kind,seed):
    if kind=='majority':
        return DummyClassifier(**DESIGN['models'][kind])
    if kind=='logistic':
        return LogisticRegression(**DESIGN['models'][kind],random_state=seed)
    return RandomForestClassifier(**DESIGN['models'][kind],random_state=seed,n_jobs=4)

def fit_or_load(path,preprocessor,matrix,y,weights,kind,seed,context,run):
    path=Path(path)
    metadata_path=path.with_suffix('.json')
    if path.exists():
        metadata=json.loads(metadata_path.read_text())
        assert metadata['fingerprint']==context['fingerprint']
        assert metadata['model_sha256']==file_hash(path)
        return joblib.load(path),metadata
    if not run:
        raise RuntimeError('No verified fit cache: '+path.name+'. Enable the appropriate run switch explicitly.')
    start=time.monotonic()
    model=build_model(kind,seed)
    with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=1):
        warnings.simplefilter('always')
        model.fit(matrix,y,sample_weight=weights*len(weights))
    warning_text=[str(w.message) for w in captured]
    metadata={'fingerprint':context['fingerprint'],'kind':kind,'fit_at_utc':timestamp(),
        'seconds':time.monotonic()-start,'training_respondents':len(y),'warnings':warning_text,
        'convergence_warning':any(issubclass(w.category,ConvergenceWarning) for w in captured)}
    bundle={'preprocessor':preprocessor,'model':model}
    temporary=path.with_suffix('.joblib.tmp')
    joblib.dump(bundle,temporary,compress=3)
    temporary.replace(path)
    metadata['model_sha256']=file_hash(path)
    json_write(metadata,metadata_path)
    return bundle,metadata

def probabilities(bundle,x,k):
    matrix=transform_predictors(x,bundle['preprocessor'])
    with threadpool_limits(limits=1):
        p=bundle['model'].predict_proba(matrix)
    assert np.array_equal(bundle['model'].classes_,np.arange(k))
    assert np.allclose(p.sum(axis=1),1)
    return p

def metrics(y,pred,weights,k,prob=None):
    cm=confusion_matrix(y,pred,labels=np.arange(k),sample_weight=weights)
    support=cm.sum(axis=1)
    recall=np.divide(np.diag(cm),support,out=np.zeros(k),where=support>0)
    precision=np.divide(np.diag(cm),cm.sum(axis=0),out=np.zeros(k),where=cm.sum(axis=0)>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(k),where=precision+recall>0)
    result={'balanced_accuracy':float(recall.mean()),'macro_f1':float(f1.mean()),
        'accuracy':float(np.trace(cm)/cm.sum())}
    if prob is not None:
        result['log_loss']=float(log_loss(y,prob,labels=np.arange(k),sample_weight=weights))
    profiles=pd.DataFrame({'cluster':np.arange(k),'precision':precision,'recall':recall,'f1':f1,
        'weighted_true_share':support/support.sum(),'weighted_predicted_share':cm.sum(axis=0)/cm.sum(),
        'respondents':[int((y==c).sum()) for c in range(k)]})
    return result,profiles,cm

def save_confusion(cm,out,model,kind,scope):
    labels=[f'{model}_C{c}' for c in range(len(cm))]
    row_normalised=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm),where=cm.sum(axis=1,keepdims=True)>0)
    pd.DataFrame(cm,index=labels,columns=labels).to_csv(out/f'{model}_{kind}_{scope}_confusion_joint.csv',index_label='true_profile')
    pd.DataFrame(row_normalised,index=labels,columns=labels).to_csv(out/f'{model}_{kind}_{scope}_confusion_recall.csv',index_label='true_profile')

def development_cv(frame,x,context,out,run=False):
    splitter=GroupKFold(n_splits=5,shuffle=True,random_state=DESIGN['seed'])
    folds=list(splitter.split(x,groups=frame.COUNTRY_ISO3))
    assignment=np.full(len(frame),-1,dtype=int)
    fold_rows=[]
    for fold,(train,validation) in enumerate(folds,1):
        train_c=set(frame.iloc[train].COUNTRY_ISO3)
        valid_c=set(frame.iloc[validation].COUNTRY_ISO3)
        assert not train_c&valid_c
        assignment[validation]=fold
        fold_rows.extend({'COUNTRY_ISO3':country,'validation_fold':fold} for country in sorted(valid_c))
    assert np.all(assignment>0)
    pd.DataFrame(fold_rows).to_csv(out/'development_country_folds.csv',index=False)
    all_metrics=[]
    pooled=[]
    profiles=[]
    warnings_and_times=[]
    for model,k in DESIGN['profile_targets'].items():
        y=frame[model+'_cluster'].to_numpy(int)
        for kind in DESIGN['models']:
            oof=np.full((len(frame),k),np.nan)
            for fold,(train,validation) in enumerate(folds,1):
                train_frame,valid_frame=frame.iloc[train],frame.iloc[validation]
                weights=country_weights(train_frame)
                prep=fit_preprocessor(x.iloc[train],weights)
                matrix=transform_predictors(x.iloc[train],prep)
                assert set(y[train])==set(range(k)) and set(y[validation])==set(range(k))
                path=out/f'{model}_{kind}_fold{fold}.joblib'
                print(f'{model} {kind}: development fold {fold}/5',flush=True)
                try:
                    bundle,info=fit_or_load(path,prep,matrix,y[train],weights,kind,
                        DESIGN['seed']+fold,context,run)
                except Exception as error:
                    json_write({'at_utc':timestamp(),'stage':'development_cv','model':model,
                        'kind':kind,'fold':fold,'error':repr(error)},out/f'failure_{model}_{kind}_fold{fold}.json')
                    raise
                warnings_and_times.append({'target':model,'classifier':kind,'fold':fold,**info})
                p=probabilities(bundle,x.iloc[validation],k)
                oof[validation]=p
                scores,_,_=metrics(y[validation],p.argmax(axis=1),country_weights(valid_frame),k,p)
                all_metrics.append({'target':model,'classifier':kind,'fold':fold,
                    'training_countries':train_frame.COUNTRY_ISO3.nunique(),
                    'validation_countries':valid_frame.COUNTRY_ISO3.nunique(),**scores})
            assert np.isfinite(oof).all()
            scores,per_profile,cm=metrics(y,oof.argmax(axis=1),country_weights(frame),k,oof)
            pooled.append({'target':model,'classifier':kind,**scores})
            profiles.append(per_profile.assign(target=model,classifier=kind,scope='development_oof'))
            save_confusion(cm,out,model,kind,'development_oof')
            predictions=frame[['WPID_RANDOM','COUNTRY_ISO3']].copy()
            predictions['true_profile']=y
            predictions['predicted_profile']=oof.argmax(axis=1)
            predictions['validation_fold']=assignment
            for c in range(k):
                predictions[f'probability_C{c}']=oof[:,c]
            predictions.to_parquet(out/f'{model}_{kind}_development_oof_predictions.parquet',index=False)
    fold_scores=pd.DataFrame(all_metrics)
    summary=fold_scores.groupby(['target','classifier'])[['balanced_accuracy','macro_f1','accuracy','log_loss']].agg(['mean','std'])
    summary.columns=['_'.join(c) for c in summary.columns]
    summary=summary.reset_index()
    summary.to_csv(out/'development_cv_summary.csv',index=False)
    fold_scores.to_csv(out/'development_fold_scores.csv',index=False)
    pd.DataFrame(pooled).to_csv(out/'development_pooled_oof_scores.csv',index=False)
    pd.concat(profiles).to_csv(out/'development_per_profile_scores.csv',index=False)
    pd.DataFrame(warnings_and_times).to_csv(out/'fit_warnings_and_times.csv',index=False)
    return summary,folds

def freeze_supervised(frame,x,summary,context,out,run=False):
    seal_path=out/'supervised_freeze_seal.json'
    preference={'logistic':0,'random_forest':1,'majority':2}
    selection={}
    for target in DESIGN['profile_targets']:
        candidates=summary.loc[summary.target.eq(target)].copy()
        candidates['preference']=candidates.classifier.map(preference)
        chosen=candidates.sort_values(['macro_f1_mean','preference'],ascending=[False,True]).iloc[0]
        selection[target]={'classifier':chosen.classifier,'development_macro_f1_mean':float(chosen.macro_f1_mean)}
    if seal_path.exists():
        seal=json.loads(seal_path.read_text())
        assert seal['fingerprint']==context['fingerprint'] and seal['selection']==selection
        for filename,digest in seal['model_hashes'].items():
            assert file_hash(out/filename)==digest
        return seal
    weights=country_weights(frame)
    prep=fit_preprocessor(x,weights)
    matrix=transform_predictors(x,prep)
    hashes={}
    for target,entry in selection.items():
        for kind in set([entry['classifier'],'majority']):
            path=out/f'{target}_{kind}_final_development.joblib'
            bundle,metadata=fit_or_load(path,prep,matrix,frame[target+'_cluster'].to_numpy(int),
                weights,kind,DESIGN['seed'],context,run)
            if metadata['convergence_warning']:
                raise RuntimeError('Final logistic regression did not converge; no supervised seal created.')
            hashes[path.name]=file_hash(path)
    seal={'fingerprint':context['fingerprint'],'selection':selection,'model_hashes':hashes,
        'frozen_at_utc':timestamp(),'heldout_predictions_used_for_selection':False,
        'profile_context':context['frozen_cluster_context'],'settings':DESIGN}
    json_write(seal,seal_path)
    return seal

def block_permutation(frame,x,folds,seal,context,out,run=False):
    result_path=out/'development_block_permutation.csv'
    if result_path.exists():
        return pd.read_csv(result_path)
    if not run:
        raise RuntimeError('Permutation result cache missing; explicitly enable RUN_INTERPRETATION.')
    records=[]
    for target,k in DESIGN['profile_targets'].items():
        kind=seal['selection'][target]['classifier']
        for fold,(_,valid) in enumerate(folds,1):
            f=frame.iloc[valid].reset_index(drop=True)
            source=x.iloc[valid].reset_index(drop=True)
            y=f[target+'_cluster'].to_numpy(int)
            weights=country_weights(f)
            bundle=joblib.load(out/f'{target}_{kind}_fold{fold}.joblib')
            base_p=probabilities(bundle,source,k)
            base=metrics(y,base_p.argmax(axis=1),weights,k)[0]['macro_f1']
            for block,columns in FEATURE_BLOCKS.items():
                for repetition in range(3):
                    rng=np.random.default_rng(DESIGN['seed']+1000*fold+repetition)
                    shuffled=source.copy()
                    if block=='Country income':
                        countries=sorted(f.COUNTRY_ISO3.unique())
                        income=[source.loc[f.COUNTRY_ISO3.eq(c),'CountryIncomeLevel'].iloc[0] for c in countries]
                        mapping=dict(zip(countries,rng.permutation(income)))
                        shuffled['CountryIncomeLevel']=f.COUNTRY_ISO3.map(mapping)
                    else:
                        for positions in f.groupby('COUNTRY_ISO3',sort=True).indices.values():
                            shuffled.iloc[positions,shuffled.columns.get_indexer(columns)]=source.iloc[rng.permutation(positions)][columns].to_numpy()
                    p=probabilities(bundle,shuffled,k)
                    score=metrics(y,p.argmax(axis=1),weights,k)[0]['macro_f1']
                    records.append({'target':target,'classifier':kind,'fold':fold,'block':block,
                        'repetition':repetition+1,'base_macro_f1':base,'permuted_macro_f1':score,
                        'macro_f1_drop':base-score,
                        'scheme':'whole-country income reassignment' if block=='Country income' else 'joint block shuffle within country'})
            print(f'{target} interpretation: fold {fold}/5 completed',flush=True)
    result=pd.DataFrame(records)
    result.to_csv(result_path,index=False)
    summary=result.groupby(['target','classifier','block']).macro_f1_drop.agg(['mean','std']).reset_index()
    summary.to_csv(out/'development_block_permutation_summary.csv',index=False)
    return result

def heldout_test(root,context,out,seal,run=False):
    complete=out/'heldout_complete.json'
    if complete.exists():
        saved=json.loads(complete.read_text())
        assert saved['fingerprint']==context['fingerprint']
        assert saved['freeze_seal_sha256']==file_hash(out/'supervised_freeze_seal.json')
        return pd.read_csv(out/'heldout_scores.csv')
    if not run:
        raise RuntimeError('No supervised held-out cache. Enable RUN_HELDOUT only after the development seal exists.')
    access_time=timestamp()
    assert access_time>seal['frozen_at_utc']
    json_write({'first_access_utc':access_time,'selection_fixed':seal['selection'],
        'supervised_freeze_sha256':file_hash(out/'supervised_freeze_seal.json')},out/'heldout_first_access.json')
    frame,inclusion=load_cohort(root,context,'heldout_country')
    x=prepare_predictors(frame)
    inclusion.to_csv(out/'heldout_target_eligibility.csv',index=False)
    predictor_missingness(frame,x,'heldout').to_csv(out/'heldout_predictor_missingness.csv',index=False)
    all_scores=[]
    all_profiles=[]
    all_countries=[]
    intervals=[]
    for target,k in DESIGN['profile_targets'].items():
        y=frame[target+'_cluster'].to_numpy(int)
        selected=seal['selection'][target]['classifier']
        for kind in dict.fromkeys(['majority',selected]):
            path=out/f'{target}_{kind}_final_development.joblib'
            assert file_hash(path)==seal['model_hashes'][path.name]
            bundle=joblib.load(path)
            p=probabilities(bundle,x,k)
            pred=p.argmax(axis=1)
            score,profiles,cm=metrics(y,pred,country_weights(frame),k,p)
            all_scores.append({'target':target,'classifier':kind,'selected':kind==selected,**score})
            all_profiles.append(profiles.assign(target=target,classifier=kind,scope='heldout'))
            save_confusion(cm,out,target,kind,'heldout')
            predictions=frame[['WPID_RANDOM','COUNTRY_ISO3']].copy()
            predictions['true_profile']=y
            predictions['predicted_profile']=pred
            for c in range(k):
                predictions[f'probability_C{c}']=p[:,c]
            predictions.to_parquet(out/f'{target}_{kind}_heldout_predictions.parquet',index=False)
            country_confusions=[]
            for country,indices in frame.groupby('COUNTRY_ISO3',sort=True).indices.items():
                w=country_weights(frame.iloc[indices])
                country_score,_,country_cm=metrics(y[indices],pred[indices],w,k,p[indices])
                present=country_cm.sum(axis=1)>0
                recall=np.divide(np.diag(country_cm),country_cm.sum(axis=1),out=np.zeros(k),where=present)
                all_countries.append({'target':target,'classifier':kind,'COUNTRY_ISO3':country,
                    'respondents':len(indices),'true_profiles_present':int(present.sum()),
                    'accuracy':country_score['accuracy'],'balanced_accuracy_present_profiles':float(recall[present].mean())})
                country_confusions.append(country_cm)
            rng=np.random.default_rng(DESIGN['seed'])
            matrices=np.array(country_confusions)
            draw_scores=[]
            for _ in range(1000):
                sampled=matrices[rng.integers(0,len(matrices),size=len(matrices))].mean(axis=0)
                support=sampled.sum(axis=1)
                recall=np.divide(np.diag(sampled),support,out=np.zeros(k),where=support>0)
                precision=np.divide(np.diag(sampled),sampled.sum(axis=0),out=np.zeros(k),where=sampled.sum(axis=0)>0)
                f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(k),where=precision+recall>0)
                draw_scores.append([recall.mean(),f1.mean(),np.trace(sampled)/sampled.sum()])
            for column,metric in enumerate(['balanced_accuracy','macro_f1','accuracy']):
                values=np.array(draw_scores)[:,column]
                intervals.append({'target':target,'classifier':kind,'metric':metric,
                    'estimate':score[metric],'p025':float(np.quantile(values,.025)),
                    'p975':float(np.quantile(values,.975)),'country_draws':1000,'models_fixed':True})
    result=pd.DataFrame(all_scores)
    result.to_csv(out/'heldout_scores.csv',index=False)
    pd.concat(all_profiles).to_csv(out/'heldout_per_profile_scores.csv',index=False)
    pd.DataFrame(all_countries).to_csv(out/'heldout_country_scores.csv',index=False)
    pd.DataFrame(intervals).to_csv(out/'heldout_country_bootstrap_intervals.csv',index=False)
    json_write({'fingerprint':context['fingerprint'],'completed_at_utc':timestamp(),
        'first_access_utc':access_time,'freeze_seal_sha256':file_hash(out/'supervised_freeze_seal.json'),
        'profile_refits':False,'retuned_after_heldout':False},complete)
    return result

def make_figures(out):
    import matplotlib.pyplot as plt
    figures=out/'figures'
    figures.mkdir(exist_ok=True)
    cv=pd.read_csv(out/'development_cv_summary.csv')
    fig,axes=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
    for ax,(target,k) in zip(axes,DESIGN['profile_targets'].items()):
        rows=cv.loc[cv.target.eq(target)].set_index('classifier').loc[list(DESIGN['models'])]
        positions=np.arange(3)
        ax.bar(positions,rows.macro_f1_mean,color=['#a5a5a5','#39789c','#6b9b77'])
        ax.errorbar(positions,rows.macro_f1_mean,yerr=rows.macro_f1_std,fmt='none',color='black',capsize=3)
        ax.set_xticks(positions,['Majority','Logistic','Random forest'])
        ax.set_ylabel('Equal-country-weighted macro-F1')
        ax.set_ylim(0,1)
        ax.set_title(f'{target}: {k} frozen profiles')
        for i,value in enumerate(rows.macro_f1_mean):ax.text(i,value+.025,f'{value:.3f}',ha='center')
    fig.suptitle('Development-only country-grouped cross-validation\nError bars are fold SD, not confidence intervals',fontsize=11)
    fig.savefig(figures/'development_classifier_comparison.png',dpi=180)
    plt.close(fig)
    seal=json.loads((out/'supervised_freeze_seal.json').read_text())
    for target,k in DESIGN['profile_targets'].items():
        kind=seal['selection'][target]['classifier']
        cm=pd.read_csv(out/f'{target}_{kind}_heldout_confusion_recall.csv',index_col=0)
        fig,ax=plt.subplots(figsize=(max(5,k*.75),max(4,k*.65)),constrained_layout=True)
        image=ax.imshow(cm,cmap='Blues',vmin=0,vmax=1)
        ax.set_xticks(range(k),cm.columns,rotation=45,ha='right')
        ax.set_yticks(range(k),cm.index)
        ax.set_xlabel('Predicted frozen profile')
        ax.set_ylabel('True frozen profile')
        ax.set_title(f'{target}: held-out recall pattern ({kind})')
        for i in range(k):
            for j in range(k):
                value=cm.iloc[i,j]
                ax.text(j,i,f'{value:.2f}',ha='center',va='center',color='white' if value>.5 else 'black',fontsize=9)
        fig.colorbar(image,ax=ax,label='Within-true-profile weighted fraction')
        fig.savefig(figures/f'{target}_heldout_confusion.png',dpi=180)
        plt.close(fig)
    importance_path=out/'development_block_permutation_summary.csv'
    if importance_path.exists():
        importance=pd.read_csv(importance_path)
        fig,axes=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
        for ax,target in zip(axes,DESIGN['profile_targets']):
            rows=importance.loc[importance.target.eq(target)].sort_values('mean')
            ax.barh(rows.block,rows['mean'],color='#39789c')
            ax.axvline(0,color='black',linewidth=.7)
            ax.set_xlabel('Development macro-F1 drop after block permutation')
            ax.set_title(target)
        fig.savefig(figures/'development_predictor_block_importance.png',dpi=180)
        plt.close(fig)
    return figures


In [ ]:
# Objective: initialise the fingerprinted result context before accessing modelling data.
CONTEXT,OUTPUT=initialise(ROOT,IMPLEMENTATION_SHA256)
display(pd.read_csv(OUTPUT/'predictor_manifest.csv')[['variable','label','block','modelling_type','missing_codes']])
print('Result folder:',OUTPUT.relative_to(ROOT))
print('Frozen profile context:',CONTEXT['frozen_cluster_context'])
print('Scientific environment:',CONTEXT['python'],CONTEXT['sklearn'],CONTEXT['pandas'])

## 2 Join frozen targets and retain explicit exclusions

**Objective:** give each eligible respondent exactly the previously frozen label, using a verified one-to-one identifier join.

Join by WPID_RANDOM, never by row order. Verify country, split and original survey-weight agreement. Keep excluded respondents in the eligibility audit, but do not impute their target labels. Both targets use the same 109,488 eligible respondents from 113 development countries. The 27,972 held-out eligible respondents from 29 countries are accessed for supervised testing only after the development selection is sealed.

The original prepared dataset contains 146,910 respondents; 9,450 have no B/E assignment. Those exclusions can affect generalizability. Objective 2 describes profile-eligible complete cases, not every survey respondent or the world population.

In [ ]:
# Objective: load development predictors and verify fixed labels without reading held-out predictors.
development,target_eligibility=load_cohort(ROOT,CONTEXT,'train')
target_eligibility.to_csv(OUTPUT/'development_target_eligibility.csv',index=False)
display(target_eligibility)
print('Eligible development respondents:',len(development))
print('Development countries:',development.COUNTRY_ISO3.nunique())
display(development[['B_profile_id','E_profile_id']].nunique().rename('distinct_profiles').to_frame())

## 3 Define external predictors and prevent target reconstruction

**Objective:** distinguish background prediction from reconstructing the clustering rule.

Use these 18 inputs:

- Age; gender; education; urban/rural residence; employment; within-country income quintile; household-size band; children in the household; marital status.
- Whether the household could cover basic needs for at least a month after losing its income.
- Perceived care from government and neighbours.
- Five discrimination questions: skin colour, religion, nationality/ethnic group, gender and disability.
- Country-income category contemporaneous with fieldwork.

The explicit allowlist excludes every profile-defining worry/harm item, transformed feature and burden score; clustering distances and margins; country profile proportions; respondent/country identifiers; and region labels as predictors. Country is used only for grouping, weights and error reporting. No published resilience composite is used because its component provenance has not yet been audited for this objective. Climate, disaster and workplace follow-ups are outside this first-draft predictor set.

Financial-buffer follow-up weeks/months have skip patterns. We use only the common parent question in this first draft; an unasked follow-up is not zero financial resilience. The combined government-care field WP22231_ALL is used instead of duplicating overlapping question versions.

Codes 98/99, demographic DK/refusal codes and genuine missing responses are non-substantive. Categorical predictors retain an explicit missing level; numeric age is imputed using the training countries' equal-country-weighted median, then scaled using their weighted mean and SD. Age has a fixed missingness indicator. Age 100 is already missing in the prepared data. Code 97 in discrimination means not applicable, not no discrimination. Country-income code 9 is retained as not classified, not imputed to a substantive income category.

All categorical features, including ordered bands, are one-hot encoded: we do not impose a linear category-spacing assumption. Encoder categories, numeric imputation and scaling are fitted separately inside each training fold. Unseen validation categories produce all-zero indicators for that feature, with no validation-based encoder refitting.

In [ ]:
# Objective: validate survey codes and display predictor missingness using development data only.
X_development=prepare_predictors(development)
missingness=predictor_missingness(development,X_development,'development')
missingness.to_csv(OUTPUT/'development_predictor_missingness.csv',index=False)
display(missingness.sort_values('equal_country_missing_fraction',ascending=False).round(4))
print('Exactly',len(PREDICTORS),'external predictors; no target-defining risk responses enter the design matrix.')

**Missingness caution:** some support/discrimination questions are absent for entire countries. Missing-category indicators may therefore carry survey-design or country-context information. That is not a leaked profile label, but it limits substantive interpretation. Country-grouped validation reduces simple memorization of respondents from the same country; it does not establish that missingness is a causal determinant of profile membership. A later sensitivity analysis can remove these indicators or question blocks.

## 4 Fix country folds and weighting

**Objective:** estimate prediction for countries outside supervised training, not for new respondents from countries already seen by the classifier.

Use five shuffled GroupKFold splits with seed 20261014. Each development country occurs in exactly one validation fold. The same country folds, eligible respondents and predictor list are used for both targets and all classifiers. No hyperparameter search or feature selection is performed in this first draft.

For a training or evaluation population containing M countries, respondent i in country c receives

$$w_{ci}=\frac{1}{M}\frac{WGT_{ci}}{\sum_{r\in c}WGT_{cr}}.$$

Recompute the denominator over eligible respondents in that training or evaluation population. Each country's weights sum to 1/M. For fitting, rescale the weights to mean one so the logistic regularization convention remains explicit; their relative values are unchanged. For reporting, retain sum-one weights. Do not concatenate split-normalized weights unchanged for a pooled analysis.

The clustering targets were discovered using all development countries. These folds evaluate prediction of that fixed taxonomy; they are not an end-to-end nested evaluation that independently rediscovers clusters inside each supervised training fold.

## 5 Compare the baseline and two classifiers

**Objective:** measure whether external characteristics improve prediction beyond the training-majority profile.

- **Majority baseline:** predict the largest equal-country-weighted training profile; its constant probability vector is the training profile distribution. Validation labels do not select the majority.
- **Multinomial logistic regression:** L2 regularization, C=1, lbfgs, tolerance 10^-6, maximum 2,000 iterations. This permits linear age effects and category-specific background associations on the log-odds scale.
- **Random forest:** 120 trees, maximum depth 14, minimum 50 training records per leaf, square-root feature sampling and ordinary tree bootstrap sampling. Original equal-country weights enter fitting. No additional class balancing is applied, so fitting weights are not silently changed.

Report equal-country-weighted balanced accuracy, macro-F1, ordinary accuracy and log loss. Balanced accuracy is the mean weighted recall across all frozen profiles. Macro-F1 gives every profile equal importance after weighted precision/recall are calculated. Classify by the largest predicted probability. Baseline balanced accuracy is 1/4 for B and 1/8 for E when all true profiles are present; high majority accuracy alone is not good multiclass prediction.

Save every validation-fold result, mean and SD, pooled out-of-fold scores, per-profile precision/recall/F1, predicted profile shares and confusion matrices. Fold SD describes geographic variability and is not a confidence interval. Pooled out-of-fold scores use equal-country weights over all 113 development countries and need not equal the unweighted mean of the five fold metrics.

The forest and logistic model are fixed comparisons, not claims of optimal performance. Convergence warnings and computational failures remain in the result folder.

In [ ]:
# Objective: fit only missing permitted caches and collect predictions from unseen development countries.
CV_SUMMARY,FOLDS=development_cv(development,X_development,CONTEXT,OUTPUT,RUN_DEVELOPMENT)
display(CV_SUMMARY.round(4))
display(pd.read_csv(OUTPUT/'development_pooled_oof_scores.csv').round(4))
display(pd.read_csv(OUTPUT/'development_per_profile_scores.csv').round(4))
display(pd.read_csv(OUTPUT/'development_country_folds.csv').groupby('validation_fold').size().rename('countries'))

## 6 Select and seal the supervised specifications

**Objective:** make the development decision before supervised held-out results are available.

Select separately for B and E using the highest mean development-fold macro-F1. An exact tie favours logistic regression, then random forest, then majority. No new threshold is inferred from the held-out data. The comparison includes the baseline so we do not assume a trained model must improve prediction.

Fit each selected classifier and its preprocessing on all eligible development respondents. Also save the training-majority comparator. Save object hashes, settings, the selection score and a UTC freeze time. Only then compute held-out predictions. The development score used to select among classifiers is a selection score, not an unbiased estimate of an extensively tuned search; there is no nested hyperparameter search here.

In [ ]:
# Objective: freeze the chosen supervised classifiers without refitting the B/E profile models.
SUPERVISED_SEAL=freeze_supervised(development,X_development,CV_SUMMARY,CONTEXT,OUTPUT,RUN_DEVELOPMENT)
display(pd.DataFrame(SUPERVISED_SEAL['selection']).T)
print('Supervised frozen at UTC:',SUPERVISED_SEAL['frozen_at_utc'])
print('Held-out predictions used for selection:',SUPERVISED_SEAL['heldout_predictions_used_for_selection'])

## 7 Interpret development predictive associations

**Objective:** identify which background blocks the selected classifier uses, without interpreting fitted importance as a causal effect.

On each out-of-fold validation set, jointly permute each block three times. For individual-level blocks, shuffle respondent rows within each validation country: this preserves country composition and relationships within the permuted block. For country-income context, reassign income categories across whole validation countries, preserving a single income category per country. Individual and country-context permutation effects therefore answer different questions and are not a causal ranking.

Record the reduction in weighted macro-F1 relative to the same unpermuted predictions. Negative values are possible. Correlated blocks can substitute for each other, so small permutation importance does not demonstrate no association. The joint government/neighbour block does not separate institutional from social effects. These development diagnostics do not change the sealed specification.

In [ ]:
# Objective: calculate or load saved development-only block diagnostics.
PERMUTATION=block_permutation(development,X_development,FOLDS,SUPERVISED_SEAL,CONTEXT,OUTPUT,RUN_INTERPRETATION)
display(pd.read_csv(OUTPUT/'development_block_permutation_summary.csv').sort_values(['target','mean'],ascending=[True,False]).round(4))

## 8 Evaluate the sealed models in held-out countries

**Objective:** test supervised prediction outside model selection while keeping the original profile targets fixed.

Apply development-fitted preprocessing and the saved selected classifiers to 27,972 eligible respondents from the 29 held-out countries. Do not train on held-out rows, refit their encoders, choose a new classifier or change thresholds after inspecting results. Report the same aggregate and per-profile metrics, alongside the development-trained majority comparator.

These countries were already used for Objective 1 transfer validation. They remain outside Objective 2 supervised selection in this draft, but are not an entirely unseen source for the whole research project. Their true labels are frozen nearest-centre assignments, not independent human assessments.

For uncertainty, use 1,000 uniform bootstrap draws of the 29 held-out countries, retaining each country's survey-weighted confusion matrix and fixed predictions. The percentile intervals describe variation over these observed countries, not worldwide population inference, model-fitting uncertainty or a full complex-survey design.

In [ ]:
# Objective: run the supervised held-out test only after the saved development seal exists.
HELDOUT=heldout_test(ROOT,CONTEXT,OUTPUT,SUPERVISED_SEAL,RUN_HELDOUT)
display(HELDOUT.round(4))
display(pd.read_csv(OUTPUT/'heldout_per_profile_scores.csv').round(4))
display(pd.read_csv(OUTPUT/'heldout_country_bootstrap_intervals.csv').round(4))
display(pd.read_csv(OUTPUT/'heldout_target_eligibility.csv'))

## 9 Inspect country errors and representation limits

**Objective:** avoid treating a reassuring global score as evidence of uniform geographic performance.

Report all 29 countries, eligible sample sizes and how many true profiles are represented. Country accuracy uses survey weights normalized within that country. Country balanced accuracy averages recall over true profiles present there; it is explicitly labelled as such because absent classes cannot have an estimated recall. It is not directly equivalent to the fixed-K aggregate balanced accuracy. Small country/profile cells warrant caution.

Also compare predictor missingness between development and held-out countries. A good aggregate result can coexist with weak prediction in particular countries or poor recall of smaller profiles. There is no automatic success threshold borrowed from the ARI 0.80 clustering-stability rule.

In [ ]:
# Objective: display the complete country-level error record and predictor-coverage audit.
country_errors=pd.read_csv(OUTPUT/'heldout_country_scores.csv')
selected_pairs={(target,value['classifier']) for target,value in SUPERVISED_SEAL['selection'].items()}
selected_countries=country_errors.loc[[pair in selected_pairs for pair in zip(country_errors.target,country_errors.classifier)]]
display(selected_countries.sort_values(['target','balanced_accuracy_present_profiles']).round(4))
display(pd.read_csv(OUTPUT/'heldout_predictor_missingness.csv').round(4))

## 10 Visual summaries and first-draft conclusions

**Objective:** connect the model-selection evidence to the held-out error patterns and qualified substantive interpretation.

The first figure compares development macro-F1, with fold SD shown explicitly. Held-out confusion matrices reveal which true profiles are confused with others. The development permutation figure summarizes block dependence under the schemes described above. These are reporting operations; no model settings are changed by drawing the figures.

In [ ]:
# Objective: save and display reproducible figures from aggregate result tables.
import matplotlib.pyplot as plt
plt.rcParams['savefig.bbox']='tight'
FIGURES=make_figures(OUTPUT)
for name in ['development_classifier_comparison.png','B_heldout_confusion.png','E_heldout_confusion.png','development_predictor_block_importance.png']:
    display(Image(filename=str(FIGURES/name)))

In [ ]:
# Objective: state the first-draft results using saved evidence, not training-set performance.
lines=[]
for target,k in DESIGN['profile_targets'].items():
    chosen=SUPERVISED_SEAL['selection'][target]['classifier']
    row=HELDOUT.loc[HELDOUT.target.eq(target)&HELDOUT.classifier.eq(chosen)].iloc[0]
    base=HELDOUT.loc[HELDOUT.target.eq(target)&HELDOUT.classifier.eq('majority')].iloc[0]
    lines.append(f"- **{target}, {k} profiles:** development-selected {chosen}; held-out balanced accuracy {row.balanced_accuracy:.3f}, macro-F1 {row.macro_f1:.3f}, accuracy {row.accuracy:.3f}. Majority balanced accuracy {base.balanced_accuracy:.3f}, macro-F1 {base.macro_f1:.3f}.")
display(Markdown("\n".join(lines)))
display(Markdown("External background characteristics predict the frozen descriptive labels to the extent shown above. Inspect per-profile recall and country errors before treating aggregate improvement as practical usefulness. These results do not establish causal explanations, validate a universal typology, or revise Objective 1."))
json_write({'status':'complete first draft','fingerprint':CONTEXT['fingerprint'],
    'selected_classifiers':SUPERVISED_SEAL['selection'],'profile_refits':False,
    'retuned_after_holdout':False,'completed_at_utc':timestamp()},OUTPUT/'first_draft_complete.json')

## First draft findings and analytical interpretation

**Development decision:** logistic regression has the highest mean country-fold macro-F1 for both targets: B 0.316, versus random forest 0.303; E 0.107, versus 0.094. That is the recorded selection criterion, not a claim that logistic regression is universally superior. The random forest has slightly lower development log loss, illustrating why the selection metric must be specified before the held-out comparison. Its fixed settings have not been optimized.

| Held-out target | Selected classifier | Balanced accuracy | Macro-F1 | Accuracy | Majority balanced accuracy |
|---|---|---:|---:|---:|---:|
| B4 | Logistic regression | 0.338 | 0.321 | 0.381 | 0.250 |
| E8 | Logistic regression | 0.151 | 0.109 | 0.357 | 0.125 |

**B4:** there is transferable background signal, but individual membership prediction remains modest. The classifier predicts the lower-worry/harm profile B_C3 for 59.1% of held-out weight, although its true share is 34.3%. B_C3 recall is 71.0%, compared with 30.4% for broad harm exposure, 22.2% for crime/weather/traffic worry and only 11.5% for food/water worry. The full weighted confusion matrix, not ordinary accuracy alone, is essential to this assessment.

**E8:** background prediction is particularly weak. The classifier assigns 87.1% of held-out weight to lower worry intensity E_C1, whose true share is 34.6%. That profile's recall is 94.6%, but traffic-focused E_C0, crime-focused E_C4 and severe-weather-focused E_C5 have zero correctly predicted held-out cases. Crime/traffic E_C3 and food/water E_C6 have recall below 1%. The aggregate improvement over a majority rule therefore does not support reliable prediction of all eight profiles. Broad-high-worry E_C7 recall is 18.9%; mental-health-related E_C2 recall is 7.1%.

The minority-profile probability estimates can be informative even when the largest-probability decision seldom selects those profiles. However, this draft has not assessed calibration, alternative decision rules or class-balanced fitting. We do not infer their effectiveness from these poor recalls. Any such comparison belongs in development-country validation, with its weighting and trade-offs recorded explicitly.

**Geographic limits:** the B classifier's held-out country balanced accuracy ranges from approximately 0.224 to 0.362; the E range is approximately 0.124 to 0.171. Thus B can fall below its 0.25 majority balanced-accuracy benchmark in a country, and several E country results are close to the 0.125 benchmark. Ordinary country accuracy is strongly affected by local profile prevalence and should not be used to rank transfer quality by itself.

**Predictor interpretation:** reassignment of country-income context and within-country discrimination-block permutation reduce development macro-F1 more than the other tested perturbations in these fitted classifiers. These effects use different permutation schemes and do not form a causal ranking. Individual-block effects are conditional on country composition; the country-income perturbation operates between countries. Correlated predictors, missing-question patterns and observational measurement limit interpretation. The financial-buffer parent question contributes little incremental hard-classification performance here; this is not evidence that detailed economic resilience has no relationship to worry or harm.

**What this says about Objective 1:** the frozen profiles can summarize reported worry/harm configurations without being strongly determined by the selected background characteristics. These results neither invalidate the descriptive clustering nor establish practical value. They show that background-based individual assignment is weak, especially for E's thematic profiles. Objective 3 still needs a separately defined external outcome and an added-value comparison.

**Validation limit:** the held-out result has now been inspected for Objective 2. Future revisions can be compared within development-country validation, but repeated selection based on these same held-out results would make them exploratory feedback rather than an untouched new test. New confirmatory claims require appropriately separated validation evidence.

**Execution record:** the first notebook launch failed before modelling because the requested local kernel alias did not exist. The runner now launches the existing scientific Python in memory, without changing the user's installed kernel. All 30 development fold fits and four final development fits completed without classifier convergence warnings or fit failures. The clustering was never rerun.


## 11 What is complete and what remains open

This draft implements the documented Week 6 comparison, development country folds, fixed preprocessing, selected-model freeze, held-out test, profile-level and country-level error reporting, development block interpretation and fixed-model country-bootstrap intervals.

Further work must have a specific question. Candidates include sensitivity to missing-category indicators, richer correctly routed financial-buffer detail, predictor-block ablations, class-balanced fitting when smaller-profile recall is inadequate, probability calibration and a development-only nested tuning comparison. None has been run or selected from this held-out performance. Any revision needs a new documented development protocol; these held-out results cannot be repeatedly used to select improvements while retaining the original test claim.

Objective 3 still needs a separately defined external outcome and overlap audit. Items used here as predictors cannot automatically become evidence of added practical value without a clearly specified outcome comparison. Keep that decision separate from this membership-prediction result.

### Reuse and privacy

The fixed-label registry, out-of-fold/held-out prediction tables and fitted executable objects contain respondent-linked or protected material. They remain local. Public reporting may use aggregate tables, figures and code, but not these protected files. Students adopting B/E should use the original identifiers and dataset fingerprint, not reverse-engineer labels from webpage numbering.

### Implementation references

- [Country-grouped cross-validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)
- [Multinomial logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Random forest classifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

Execution uses the installed versions recorded in the context, not an assumed match to the evolving online documentation.